# 96 — GNN Edge Classifier Test

Heterogeneous GNN (`HeteroConv` + `NNConv`) for directed attack-edge classification.
2-planet scenario: one owned by player 0, one neutral.
Label = 1 if my ships > neutral ships at step 0.

In [ ]:
%run 96-library.py
import torch
import torch.nn as nn
from torch_geometric.nn import SAGEConv, NNConv
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
import copy, math, random

In [ ]:
_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}

def make_animation(snapshots, title='', interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')
    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100); ax.set_ylim(100, 0)
        ax.set_aspect('equal'); ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values(): sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y,   str(ships),         ha='center', va='center', color='white', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y+2, str(pid),            ha='center', va='center', color='red',   fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y-2, '+'+str(production), ha='center', va='center', color='white', fontsize=5, fontweight='bold', zorder=4)
        for f in snap['fleets']:
            fid, owner, x, y, ang, from_id, ships = f
            ax.plot(x, y, 'D', color=_COLORS.get(owner,'#888888'), markersize=5, zorder=5)
        return []
    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

In [ ]:
data0, snaps0 = generate_sample(42)
print("master x:", data0['master'].x.shape)
print("planet x:", data0['planet'].x.shape)
print("attack edge_index:", data0['planet','attacks','planet'].edge_index.shape)
print("attack edge_attr: ", data0['planet','attacks','planet'].edge_attr.shape)
print("label:", int(data0.y.item()))
make_animation(snaps0, title='Sample seed=42', interval=200)

In [ ]:
print("Generating train dataset (100 samples)...")
train_dataset = [generate_sample(i)      for i in range(100)]
print("Generating test dataset (10 samples)...")
test_dataset  = [generate_sample(1000+i) for i in range(10)]
train_labels = [int(d.y.item()) for d, _ in train_dataset]
test_labels  = [int(d.y.item()) for d, _ in test_dataset]
print(f"Train label distribution: {train_labels.count(1)} pos / {train_labels.count(0)} neg")
print(f"Test  label distribution: {test_labels.count(1)} pos / {test_labels.count(0)} neg")

In [ ]:
class GNNEdgeClassifier(nn.Module):
    """2-layer heterogeneous GNN for directed edge classification.

    Node types: master (6-dim), planet (22-dim)
    Edge types:
      (planet, to_master, master) -- SAGEConv, no edge features
      (master, to_planet, planet) -- SAGEConv, no edge features
      (planet, attacks,   planet) -- NNConv, 50-dim edge features
    """
    def __init__(self, hidden_dim: int = 64):
        super().__init__()
        H = hidden_dim

        self.master_lin = nn.Linear(6,  H)
        self.planet_lin = nn.Linear(22, H)

        # Layer 1
        self.sage1_pm = SAGEConv(H, H)
        self.sage1_mp = SAGEConv(H, H)
        self.nnconv1  = NNConv(H, H, nn=nn.Linear(50, H * H))

        # Layer 2
        self.sage2_pm = SAGEConv(H, H)
        self.sage2_mp = SAGEConv(H, H)
        self.nnconv2  = NNConv(H, H, nn=nn.Linear(50, H * H))

        # Edge classifier: [h_src | h_dst | edge_attr] -> logit
        self.edge_mlp = nn.Sequential(
            nn.Linear(H * 2 + 50, H),
            nn.ReLU(),
            nn.Linear(H, 1),
        )

    def _pass(self, h_p, h_m, data, sage_pm, sage_mp, nnconv):
        ei_pm = data['planet', 'to_master', 'master'].edge_index
        ei_mp = data['master', 'to_planet', 'planet'].edge_index
        ei_pp = data['planet', 'attacks',   'planet'].edge_index
        ea_pp = data['planet', 'attacks',   'planet'].edge_attr
        new_m = sage_pm((h_p, h_m), ei_pm)
        new_p = sage_mp((h_m, h_p), ei_mp) + nnconv(h_p, ei_pp, ea_pp)
        return torch.relu(new_p), torch.relu(new_m)

    def forward(self, data):
        h_m = torch.relu(self.master_lin(data['master'].x))
        h_p = torch.relu(self.planet_lin(data['planet'].x))
        h_p, h_m = self._pass(h_p, h_m, data, self.sage1_pm, self.sage1_mp, self.nnconv1)
        h_p, h_m = self._pass(h_p, h_m, data, self.sage2_pm, self.sage2_mp, self.nnconv2)
        ei = data['planet', 'attacks', 'planet'].edge_index
        ea = data['planet', 'attacks', 'planet'].edge_attr
        edge_in = torch.cat([h_p[ei[0]], h_p[ei[1]], ea], dim=-1)
        return self.edge_mlp(edge_in).squeeze(-1)


model = GNNEdgeClassifier(hidden_dim=64)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# Shape check on sample 0
data0, _ = train_dataset[0]
logit = model(data0)
assert logit.shape == (1,), f"Expected (1,), got {logit.shape}"
print(f"Forward pass OK -- logit: {logit.item():.4f}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

model.train()
for epoch in range(50):
    total_loss = 0.0
    for data, _ in train_dataset:
        optimizer.zero_grad()
        logit = model(data)
        loss  = criterion(logit, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | avg loss {total_loss / len(train_dataset):.4f}")

print("Training complete.")

In [ ]:
model.eval()
preds, labels_list = [], []
with torch.no_grad():
    for data, _ in test_dataset:
        logit = model(data)
        pred  = int((torch.sigmoid(logit) > 0.5).item())
        preds.append(pred)
        labels_list.append(int(data.y.item()))

print(f"Test Accuracy: {accuracy_score(labels_list, preds):.2f}\n")
print(classification_report(labels_list, preds, target_names=['no-attack','attack']))

In [ ]:
i = 0
data_i, snaps_i = test_dataset[i]
with torch.no_grad():
    pred_i = int((torch.sigmoid(model(data_i)) > 0.5).item())
true_i = int(data_i.y.item())
print(f"Sample {i}: pred={pred_i}  true={true_i}  {'OK' if pred_i==true_i else 'WRONG'}")
make_animation(snaps_i, title=f'Test {i} | pred={pred_i} true={true_i}', interval=200)

In [ ]:
i = 1
data_i, snaps_i = test_dataset[i]
with torch.no_grad():
    pred_i = int((torch.sigmoid(model(data_i)) > 0.5).item())
true_i = int(data_i.y.item())
print(f"Sample {i}: pred={pred_i}  true={true_i}  {'OK' if pred_i==true_i else 'WRONG'}")
make_animation(snaps_i, title=f'Test {i} | pred={pred_i} true={true_i}', interval=200)

In [ ]:
i = 2
data_i, snaps_i = test_dataset[i]
with torch.no_grad():
    pred_i = int((torch.sigmoid(model(data_i)) > 0.5).item())
true_i = int(data_i.y.item())
print(f"Sample {i}: pred={pred_i}  true={true_i}  {'OK' if pred_i==true_i else 'WRONG'}")
make_animation(snaps_i, title=f'Test {i} | pred={pred_i} true={true_i}', interval=200)

In [ ]:
i = 3
data_i, snaps_i = test_dataset[i]
with torch.no_grad():
    pred_i = int((torch.sigmoid(model(data_i)) > 0.5).item())
true_i = int(data_i.y.item())
print(f"Sample {i}: pred={pred_i}  true={true_i}  {'OK' if pred_i==true_i else 'WRONG'}")
make_animation(snaps_i, title=f'Test {i} | pred={pred_i} true={true_i}', interval=200)

In [ ]:
i = 4
data_i, snaps_i = test_dataset[i]
with torch.no_grad():
    pred_i = int((torch.sigmoid(model(data_i)) > 0.5).item())
true_i = int(data_i.y.item())
print(f"Sample {i}: pred={pred_i}  true={true_i}  {'OK' if pred_i==true_i else 'WRONG'}")
make_animation(snaps_i, title=f'Test {i} | pred={pred_i} true={true_i}', interval=200)

In [ ]:
i = 5
data_i, snaps_i = test_dataset[i]
with torch.no_grad():
    pred_i = int((torch.sigmoid(model(data_i)) > 0.5).item())
true_i = int(data_i.y.item())
print(f"Sample {i}: pred={pred_i}  true={true_i}  {'OK' if pred_i==true_i else 'WRONG'}")
make_animation(snaps_i, title=f'Test {i} | pred={pred_i} true={true_i}', interval=200)

In [ ]:
i = 6
data_i, snaps_i = test_dataset[i]
with torch.no_grad():
    pred_i = int((torch.sigmoid(model(data_i)) > 0.5).item())
true_i = int(data_i.y.item())
print(f"Sample {i}: pred={pred_i}  true={true_i}  {'OK' if pred_i==true_i else 'WRONG'}")
make_animation(snaps_i, title=f'Test {i} | pred={pred_i} true={true_i}', interval=200)

In [ ]:
i = 7
data_i, snaps_i = test_dataset[i]
with torch.no_grad():
    pred_i = int((torch.sigmoid(model(data_i)) > 0.5).item())
true_i = int(data_i.y.item())
print(f"Sample {i}: pred={pred_i}  true={true_i}  {'OK' if pred_i==true_i else 'WRONG'}")
make_animation(snaps_i, title=f'Test {i} | pred={pred_i} true={true_i}', interval=200)

In [ ]:
i = 8
data_i, snaps_i = test_dataset[i]
with torch.no_grad():
    pred_i = int((torch.sigmoid(model(data_i)) > 0.5).item())
true_i = int(data_i.y.item())
print(f"Sample {i}: pred={pred_i}  true={true_i}  {'OK' if pred_i==true_i else 'WRONG'}")
make_animation(snaps_i, title=f'Test {i} | pred={pred_i} true={true_i}', interval=200)

In [ ]:
i = 9
data_i, snaps_i = test_dataset[i]
with torch.no_grad():
    pred_i = int((torch.sigmoid(model(data_i)) > 0.5).item())
true_i = int(data_i.y.item())
print(f"Sample {i}: pred={pred_i}  true={true_i}  {'OK' if pred_i==true_i else 'WRONG'}")
make_animation(snaps_i, title=f'Test {i} | pred={pred_i} true={true_i}', interval=200)